# SMH 相对 SOXX 前日 20 日领导性：Qlib 来源映射与旧 XSD 去重

冻结来源哈希、09:46 前日时钟、原 Stage1 39 项与已关闭 XSD 信号。只做训练期去重，零未来标签、零拟合、零账户。

In [1]:
import os,sys,json,hashlib,io,zipfile,datetime,re,tempfile
from pathlib import Path
from xml.etree import ElementTree as ET
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_settings
card=root/'research/stage1_smh_soxx_leadership_qlib_map_20261023';plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();src=plan['source'];q=plan['calendar_and_price'];parent=plan['parent_original39'];xsd=plan['closed_XSD_map']
assert sha(root/src['card']/'summary.json')==src['source_summary_sha256'] and sha(root/src['raw_workbook'])==src['raw_sha256']
assert json.loads((root/src['card']/'nas_source_archive.json').read_text())['sha256']==src['nas_sha256']
assert sha(root/q['provider']/'qlib.yaml')==q['qlib_yaml_sha256'] and sha(root/q['provider']/'features/soxx/close.day.bin')==q['soxx_close_sha256']
pa=root/parent['card']/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts';assert all(sha(pa/n)==parent[k] for n,k in [('models/base_outer_71','base_model_artifact_sha256'),('inputs/features','base_features_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256')])
xa=root/xsd['card']/'qlib_recorder'/xsd['experiment_id']/xsd['recorder_id']/'artifacts/analysis/mapped_signals';assert sha(xa)==xsd['mapped_signal_artifact_sha256']
state={'plan_sha256':sha(card/'plan.json'),'phase':'source_map_attempted','raw_workbook_load_attempts':1,'D_calendar_call_attempts':1,'D_features_call_attempts':1,'parent_Recorder_object_load_attempts':3,'XSD_Recorder_object_load_attempts':1,'DatasetH_build_attempts':1,'new_Qlib_Recorder_attempts':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
raw=(root/src['raw_workbook']).read_bytes();assert hashlib.sha256(raw).hexdigest()==src['raw_sha256'];ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 shared=ET.fromstring(z.read('xl/sharedStrings.xml'));strings=[''.join(t.text or '' for t in si.findall('.//m:t',ns)) for si in shared.findall('m:si',ns)]
 sheet=ET.fromstring(z.read('xl/worksheets/sheet1.xml'));rows=sheet.findall('.//m:sheetData/m:row',ns)
 def cells(row):
  out={}
  for cell in row.findall('m:c',ns):
   col=re.match(r'[A-Z]+',cell.attrib['r']).group();v=cell.findtext('m:v',default='',namespaces=ns);out[col]=strings[int(v)] if cell.attrib.get('t')=='s' and v else v
  return out
 assert [cells(rows[1]).get(k) for k in 'ABE']==['Date','NAV','Last Trade']
 smh={}
 for row in rows[2:]:
  v=cells(row)
  try:d=pd.Timestamp(datetime.datetime.strptime(v.get('A',''),'%m/%d/%Y').date())
  except ValueError:continue
  value=float(v['E'].replace(',',''));assert np.isfinite(value) and value>0 and d not in smh;smh[d]=value
assert len(smh)==4090
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/q['provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time=q['calendar_bounds'][0],end_time=q['calendar_bounds'][1],freq='day'));assert len(cal)==3964 and cal.is_unique
soxx_raw=D.features(['SOXX'],['$close'],start_time=q['calendar_bounds'][0],end_time=q['calendar_bounds'][1],freq='day')
soxx=soxx_raw['$close'].droplevel('instrument').reindex(cal).astype(float);assert np.isfinite(soxx).all() and (soxx>0).all()
smh_close=pd.Series(smh,dtype=float).reindex(cal);assert (smh_close.dropna()>0).all()
mapped=np.log(smh_close.shift(1)/smh_close.shift(21))-np.log(soxx.shift(1)/soxx.shift(21));mapped.name='SMH_minus_SOXX_prior_market_20'
with R.uri_context(str(root/parent['card']/'qlib_recorder')):
 pr=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=pr.load_object('models/base_outer_71');base=pr.load_object('inputs/features');frame=pr.load_object('inputs/daily_frame')
base_names=list(original.base_features);assert len(base_names)==39 and len(set(base_names))==39 and set(base_names).issubset(base.columns) and base.index.equals(frame.index)
with R.uri_context(str(root/xsd['card']/'qlib_recorder')):
 xr=R.get_recorder(recorder_id=xsd['recorder_id'],experiment_id=xsd['experiment_id']);xs=xr.load_object('analysis/mapped_signals')
assert cal.equals(pd.DatetimeIndex(xs.index)) and 'XSD_minus_SOXX_log_total_return_20' in xs.columns
train=cal[(cal>=pd.Timestamp(plan['cohorts']['train'][0]))&(cal<=pd.Timestamp(plan['cohorts']['train'][1]))];held=cal[(cal>=pd.Timestamp(plan['cohorts']['held_reused'][0]))&(cal<=pd.Timestamp(plan['cohorts']['held_reused'][1]))]
assert len(train)==3020 and len(held)==944
eligible=mapped.loc[train].dropna().index;held_eligible=mapped.loc[held].dropna().index
assert len(eligible)>=plan['cohorts']['train_min_known'] and len(held_eligible)==plan['cohorts']['held_expected_known']
corr=base.loc[eligible,base_names].astype(float).corrwith(mapped.loc[eligible]).dropna().sort_values(key=lambda v:v.abs(),ascending=False)
xsd_corr=float(mapped.loc[eligible].corr(xs.loc[eligible,'XSD_minus_SOXX_log_total_return_20']))
maxabs=float(corr.abs().max());duplicate=bool(maxabs>=plan['redundancy_gate']['max_abs_original39_pairwise_Pearson_below'] or abs(xsd_corr)>=plan['redundancy_gate']['abs_Pearson_with_closed_XSD_signal_below'])
static=pd.concat({'feature':mapped.to_frame(),'meta':pd.DataFrame({'source_known':mapped.notna()},index=cal)},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['SOXX']*len(cal)],names=['datetime','instrument'])
ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train']),'held':tuple(plan['cohorts']['held_reused'])});assert len(ds.prepare('train',col_set=['feature','meta']))==3020 and len(ds.prepare('held',col_set=['feature','meta']))==944
report={'decision':'close_duplicate_before_fit' if duplicate else 'source_map_novelty_screen_passed_conditional_retrospective_only','plan_sha256':sha(card/'plan.json'),'calendar_rows':len(cal),'train_rows':len(train),'train_known':len(eligible),'train_unknown':len(train)-len(eligible),'held_rows':len(held),'held_known':len(held_eligible),'held_unknown':len(held)-len(held_eligible),'max_abs_original39_train_Pearson':maxabs,'top_original39_train_correlations':[{'feature':str(k),'pearson':float(v)} for k,v in corr.head(10).items()],'closed_XSD_train_Pearson':xsd_corr,'duplicate_by_frozen_gate':duplicate,'first_public_clock_qualified':False,'dividend_and_adjustment_qualified':False,'market_future_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'preflight.json').write_text(json.dumps(report,indent=2)+'\n')
state['new_Qlib_Recorder_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds);rec.save_objects(artifact_path='analysis',mapped_signals=pd.DataFrame({'SMH_minus_SOXX_prior_market_20':mapped,'source_known':mapped.notna()},index=cal),preflight=report,training_feature_correlations=corr);rec.log_metrics(train_known=len(eligible),held_known=len(held_eligible),max_abs_original39_Pearson=maxabs,XSD_Pearson=xsd_corr,duplicate=int(duplicate),model_fits=0,native_accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'preflight':'preflight.json','model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Conditional retrospective source map only: SMH issuer historical delivery and revisions are unqualified.','SMH prior close and Qlib SOXX adjusted close include different dividend/price planes.','No future outcome, information fit or Stage1 account.']}
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='stage1-smh-map-verify-') as tmp:
 restored=materialize_nas_evidence(binding,card=Path(tmp));assert restored['archive_sha256']==summary['nas_evidence']['sha256']
 with R.uri_context(str(Path(tmp)/'qlib_recorder')):
  check=R.get_recorder(recorder_id=rid,experiment_id=eid);assert check.status=='FINISHED' and check.load_object('analysis/preflight')==report
summary['nas_readback']={'verified':True,'sha256':summary['nas_evidence']['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';state['recorder_id']=rid;state['nas_sha256']=summary['nas_evidence']['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:report[k] for k in ('decision','train_known','held_known','max_abs_original39_train_Pearson','closed_XSD_train_Pearson','duplicate_by_frozen_gate')})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[36724:MainThread](2026-10-23 08:54:05,851) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[36724:MainThread](2026-10-23 08:54:05,853) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[36724:MainThread](2026-10-23 08:54:05,854) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

[36724:MainThread](2026-10-23 08:54:06,436) INFO - qlib.workflow - [exp.py:258] - Experiment 674590108897401409 starts running ...


[36724:MainThread](2026-10-23 08:54:06,488) INFO - qlib.workflow - [recorder.py:345] - Recorder 2c50e8861fda49ce8360def6f4c6dc25 starts running under Experiment 674590108897401409 ...


[36724:MainThread](2026-10-23 08:54:07,380) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'decision': 'source_map_novelty_screen_passed_conditional_retrospective_only', 'train_known': 2755, 'held_known': 944, 'max_abs_original39_train_Pearson': 0.21583259559850754, 'closed_XSD_train_Pearson': -0.13908919026862243, 'duplicate_by_frozen_gate': False}


# SMH 相对 SOXX 领导性：Qlib 映射后关闭

原生 Qlib 日历、SOXX 收盘价、SMH 发行人文件、冻结 Stage1 原39项与已关闭 XSD Recorder 已按哈希匹配。训练 3,020 日中 SMH 窗口可用 **2,755 日**，其余 265 日主要为 SMH 2011-12 上市前及 20 日窗口；复用留出 944 日全部可用。训练期 SMH 坐标与原39项最大绝对 Pearson 为 **0.216**，与已关闭 XSD 相对信号为 **−0.139**。Qlib Recorder `2c50e8861fda49ce8360def6f4c6dc25` 已完成，NAS SHA-256 `a512543c…` 已限定恢复；零未来目标、零拟合、零账户。

数值去重门虽通过，[XSD 固定信息检验结论](../stage1_xsd_breadth_h5_information_20261023/conclusion.md)已明确禁止通过换 ETF 重试同类相对领导性。低相关只是必要诊断，不自动使换 ETF 成为新的独立经济机制；2023–2026 留出也早已反复使用。`semantic_route_review.json` 因此将 SMH 版本 **在拟合前关闭**，不做 H5 MSE 或 0bp 账户。

SMH 原件历史首次送达、分红与 SOXX Qlib 调整价的同平面性亦未认证。下一步回到完整 Stage1 0bp 差区间的现金与 SOXL 选择归因，找与 ETF 领导性、发行人份额、季度利润扩散和已失败风险路线实质不同、具有决策时可用源的机制。SOXX 费用后稳定超额、2/5bp 与独立前向门仍未通过。
